In [2]:
"""
Synthetic KPI Dashboard Data Pipeline (CSV ONLY)

Step 1: Generate RAW data -> data_raw/*.csv
Step 2: Pre-clean / standardize -> data_curated/*.csv

No parquet outputs.
"""

import os
import numpy as np
import pandas as pd
!pip install faker
from faker import Faker
from datetime import datetime, timedelta, timezone

# -----------------------
# Config
# -----------------------
SEED = 42
fake = Faker()
rng = np.random.default_rng(SEED)

REGIONS = ["East", "West", "Central", "North", "South"]
CHANNEL_CANON = ["email", "social", "referral", "search", "display"]
TOUCHPOINTS = ["online_inquiry", "call", "showroom_visit", "test_drive"]

BASE_START = datetime(2024, 1, 1, tzinfo=timezone.utc)
BASE_END = datetime(2025, 12, 31, tzinfo=timezone.utc)


def ensure_dirs():
    os.makedirs("data_raw", exist_ok=True)
    os.makedirs("data_curated", exist_ok=True)


# -----------------------
# Helpers (raw)
# -----------------------
def messy_channel(ch: str) -> str:
    variants = {
        "email": ["Email", "EDM", "newsletter", "e-mail"],
        "social": ["Paid Social", "social", "FB", "IG", "Social Ads"],
        "referral": ["referral", "partner_referral", "dealer_referral"],
        "search": ["Search", "SEM", "Paid Search", "Google Ads"],
        "display": ["Display", "programmatic", "banner"],
    }
    return rng.choice(variants[ch])


def messy_status(system: str) -> str:
    # simulate inconsistent dealer status systems
    if system == "temp_scale":
        return rng.choice(["Hot", "Warm", "Cold", "Inactive"])
    return rng.choice(["Open", "In progress", "Closed", "New"])


def random_dt(start: datetime, end: datetime) -> pd.Timestamp:
    delta = (end - start).total_seconds()
    return pd.Timestamp(start + timedelta(seconds=float(rng.random() * delta)))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.1 MB/s eta 0:00:00


In [3]:
# -----------------------
# Step 1: Generate RAW (CSV only)
# -----------------------
def generate_raw(
    n_dealers: int = 60,
    n_campaigns: int = 30,
    n_models: int = 12,
    n_leads: int = 200_000,
):
    """
    Output (CSV only):
      data_raw/dealers.csv
      data_raw/models.csv
      data_raw/campaigns.csv
      data_raw/leads.csv
      data_raw/interactions.csv
      data_raw/outcomes.csv
    """
    ensure_dirs()

    # dealers (anonymous)
    dealers = pd.DataFrame({
        "dealer_id": [f"D{str(i).zfill(3)}" for i in range(1, n_dealers + 1)],
        "dealer_name": [f"Dealer_{str(i).zfill(3)}" for i in range(1, n_dealers + 1)],
        "region": rng.choice(REGIONS, size=n_dealers),
        "dealer_group": rng.choice(["Group_A", "Group_B", "Group_C", "Independent"], size=n_dealers),
        # used only for generating inconsistent raw status
        "status_system": rng.choice(["temp_scale", "pipeline"], size=n_dealers),
    })

    # models (anonymous)
    models = pd.DataFrame({
        "model_id": [f"M{str(i).zfill(2)}" for i in range(1, n_models + 1)],
        "model_name": [f"Model_{chr(64+i)}" for i in range(1, n_models + 1)],
        "segment": rng.choice(["SUV", "Sedan", "EV", "Coupe"], size=n_models),
        "price_band": rng.choice(["entry", "mid", "premium"], size=n_models, p=[0.35, 0.45, 0.20]),
    })

    # campaigns
    campaign_start = [random_dt(BASE_START, BASE_END - timedelta(days=30)).to_pydatetime() for _ in range(n_campaigns)]
    campaign_end = [s + timedelta(days=int(rng.integers(14, 90))) for s in campaign_start]

    campaigns = pd.DataFrame({
        "campaign_id": [f"C{str(i).zfill(3)}" for i in range(1, n_campaigns + 1)],
        "campaign_name": [f"Campaign_{i:02d}" for i in range(1, n_campaigns + 1)],
        "channel": rng.choice(CHANNEL_CANON, size=n_campaigns),
        "start_date": pd.to_datetime(campaign_start),
        "end_date": pd.to_datetime(campaign_end),
        "spend": np.round(rng.gamma(shape=4.0, scale=5000.0, size=n_campaigns), 2),
    })

    # leads
    lead_ids = np.arange(1, n_leads + 1)
    dealer_idx = rng.integers(0, n_dealers, size=n_leads)
    model_idx = rng.integers(0, n_models, size=n_leads)
    camp_idx = rng.integers(0, n_campaigns, size=n_leads)

    created_at = pd.to_datetime([random_dt(BASE_START, BASE_END) for _ in range(n_leads)], utc=True)

    status_system = dealers.loc[dealer_idx, "status_system"].to_numpy()
    lead_status_raw = [messy_status(sys) for sys in status_system]

    camp_channel = campaigns.loc[camp_idx, "channel"].to_numpy()
    channel_raw = [messy_channel(ch) for ch in camp_channel]

    leads = pd.DataFrame({
        "lead_id": lead_ids,
        "dealer_id": dealers.loc[dealer_idx, "dealer_id"].to_numpy(),
        "model_id": models.loc[model_idx, "model_id"].to_numpy(),
        "campaign_id": campaigns.loc[camp_idx, "campaign_id"].to_numpy(),
        "created_at": created_at,
        "customer_type": rng.choice(["new", "returning"], size=n_leads, p=[0.78, 0.22]),
        "lead_status_raw": lead_status_raw,
        "channel_raw": channel_raw,
    })

    # interactions (with duplicates + timezone noise)
    n_int = rng.poisson(lam=2.0, size=n_leads)
    n_int = np.clip(n_int, 0, 6)

    interaction_rows = int(n_int.sum())
    lead_rep = np.repeat(leads["lead_id"].to_numpy(), n_int)
    lead_created_rep = np.repeat(leads["created_at"].to_numpy(), n_int)

    offset_minutes = rng.integers(1, 60 * 24 * 10, size=interaction_rows)
    ts = pd.to_datetime(lead_created_rep) + pd.to_timedelta(offset_minutes, unit="m")

    dup_mask = rng.random(interaction_rows) < 0.05
    dup_leads = lead_rep[dup_mask]
    dup_ts = ts[dup_mask] + pd.to_timedelta(rng.integers(2, 6, size=dup_mask.sum()), unit="m")

    interactions = pd.DataFrame({
        "interaction_id": np.arange(1, interaction_rows + 1),
        "lead_id": lead_rep,
        "touchpoint": rng.choice(TOUCHPOINTS, size=interaction_rows, p=[0.45, 0.30, 0.15, 0.10]),
        "ts": ts,
        "outcome_raw": rng.choice(["contacted", "no_answer", "left_message"], size=interaction_rows, p=[0.55, 0.30, 0.15]),
        "timezone_raw": rng.choice(["UTC", "LOCAL"], size=interaction_rows, p=[0.6, 0.4]),
    })

    dups = pd.DataFrame({
        "interaction_id": np.arange(interaction_rows + 1, interaction_rows + 1 + dup_mask.sum()),
        "lead_id": dup_leads,
        "touchpoint": rng.choice(TOUCHPOINTS, size=dup_mask.sum()),
        "ts": dup_ts,
        "outcome_raw": rng.choice(["contacted", "no_answer"], size=dup_mask.sum(), p=[0.6, 0.4]),
        "timezone_raw": rng.choice(["UTC", "LOCAL"], size=dup_mask.sum(), p=[0.5, 0.5]),
    })

    interactions = pd.concat([interactions, dups], ignore_index=True)

    # outcomes (synthetic)
    first_ts = interactions.groupby("lead_id")["ts"].min()
    lead_created = leads.set_index("lead_id")["created_at"]
    response_minutes = ((first_ts - lead_created).dt.total_seconds() / 60).reindex(leads["lead_id"]).fillna(np.inf)

    has_interaction = np.isfinite(response_minutes.to_numpy())
    fast = response_minutes.to_numpy() <= 60

    base_win = 0.06
    win_prob = base_win + 0.06 * has_interaction + 0.05 * fast
    win_prob = np.clip(win_prob, 0.01, 0.30)

    won = rng.random(n_leads) < win_prob
    lost = (~won) & (rng.random(n_leads) < 0.55)
    final_status = np.where(won, "won", np.where(lost, "lost", "open"))

    conv_days = rng.integers(1, 45, size=n_leads)
    conversion_date = pd.to_datetime(leads["created_at"]) + pd.to_timedelta(conv_days, unit="D")
    conversion_date = conversion_date.where(won, pd.NaT)

    deal_value = np.round(rng.lognormal(mean=10.6, sigma=0.35, size=n_leads), 2)
    deal_value = np.where(won, deal_value, 0.0)

    outcomes = pd.DataFrame({
        "lead_id": leads["lead_id"],
        "final_status": final_status,
        "conversion_date": conversion_date,
        "deal_value": deal_value,
    })

    # save RAW CSV only (drop status_system before saving)
    dealers_out = dealers.drop(columns=["status_system"])

    dealers_out.to_csv("data_raw/dealers.csv", index=False)
    models.to_csv("data_raw/models.csv", index=False)
    campaigns.to_csv("data_raw/campaigns.csv", index=False)
    leads.to_csv("data_raw/leads.csv", index=False)
    interactions.to_csv("data_raw/interactions.csv", index=False)
    outcomes.to_csv("data_raw/outcomes.csv", index=False)

    print("✅ RAW generated (CSV only)")
    print("leads:", len(leads), "interactions:", len(interactions), "outcomes:", len(outcomes))


In [4]:
# -----------------------
# Step 2: Pre-clean / standardize (CSV only)
# -----------------------
def std_channel(x: str) -> str:
    x = str(x).lower()
    if "email" in x or "edm" in x or "newsletter" in x or "e-mail" in x:
        return "email"
    if "social" in x or x in ["fb", "ig"] or "ads" in x:
        return "social"
    if "referral" in x or "partner" in x:
        return "referral"
    if "search" in x or "sem" in x or "google" in x:
        return "search"
    if "display" in x or "banner" in x or "programmatic" in x:
        return "display"
    return "other"


def std_stage(x: str) -> str:
    x = str(x).lower()
    if x in ["hot", "warm"]:
        return "qualified"
    if x in ["cold", "inactive"]:
        return "unqualified"
    if x in ["open", "new", "in progress"]:
        return "open"
    if x in ["closed"]:
        return "closed"
    return "unknown"


def pre_clean(dedup_window_seconds: int = 300):
    """
    Input: data_raw/*.csv
    Output (CSV only):
      data_curated/leads_curated.csv
      data_curated/interactions_curated.csv
      data_curated/outcomes.csv
    """
    ensure_dirs()

    leads = pd.read_csv("data_raw/leads.csv")
    interactions = pd.read_csv("data_raw/interactions.csv")
    outcomes = pd.read_csv("data_raw/outcomes.csv")

    # 1) standardize leads
    leads_cur = leads.copy()
    leads_cur["created_at"] = pd.to_datetime(leads_cur["created_at"], utc=True, errors="coerce")
    leads_cur["channel_std"] = leads_cur["channel_raw"].apply(std_channel)
    leads_cur["lead_stage_std"] = leads_cur["lead_status_raw"].apply(std_stage)

    # 2) clean interactions: parse ts + dedup within time window
    inter_cur = interactions.copy()
    inter_cur["ts"] = pd.to_datetime(inter_cur["ts"], utc=True, errors="coerce")

    inter_cur = inter_cur.sort_values(["lead_id", "touchpoint", "ts"])
    time_diff = inter_cur.groupby(["lead_id", "touchpoint"])["ts"].diff().dt.total_seconds().fillna(999999)
    inter_cur = inter_cur[(time_diff > dedup_window_seconds) | (time_diff == 999999)].copy()

    # 3) outcomes: parse conversion_date (optional)
    out_cur = outcomes.copy()
    out_cur["conversion_date"] = pd.to_datetime(out_cur["conversion_date"], utc=True, errors="coerce")

    # save curated CSV only
    leads_cur.to_csv("data_curated/leads_curated.csv", index=False)
    inter_cur.to_csv("data_curated/interactions_curated.csv", index=False)
    out_cur.to_csv("data_curated/outcomes.csv", index=False)

    print("✅ CURATED saved (CSV only)")
    print("interactions before:", len(interactions), "after:", len(inter_cur))


# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    # Step 1: raw
    generate_raw(
        n_dealers=60,
        n_campaigns=30,
        n_models=12,
        n_leads=200_000,  # 想要 1M 就改这里
    )

    # Step 2: pre-clean
    pre_clean(dedup_window_seconds=300)

    # stop here: only raw + curated

✅ RAW generated (CSV only)
leads: 200000 interactions: 418695 outcomes: 200000
✅ CURATED saved (CSV only)
interactions before: 418695 after: 413607
